# Lab 6: Agentic RAG with LangGraph

**Level:** Challenge | **Duration:** ~45 minutes

## What You'll Learn
- Why "always retrieve" is wasteful and sometimes harmful
- How to build a RAG agent that decides WHEN to retrieve
- How to implement self-reflection: evaluating whether retrieved context is sufficient
- How to use LangGraph for stateful agent loops
- How agentic RAG compares to always-retrieve RAG

## Architecture
```
Question --> [Router] --> needs retrieval? --YES--> [Retrieve] --> [Evaluate] --> good enough? --NO--> [Re-retrieve/Refine]
                |                                                                    |                         |
                NO                                                                  YES                        |
                |                                                                    |                         |
                v                                                                    v                         v
           [Direct Answer]                                                    [Generate Answer]         [Generate Answer]
```

This is a Challenge lab. You'll get architectural guidance and key code, but you'll need to fill in some pieces yourself.

## Setup

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb langgraph

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

from typing import TypedDict, Annotated, Literal
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

print("Setup complete!")

## Step 1: Knowledge Base

Same airline operations domain, but the agent must decide whether to consult it.

In [ ]:
knowledge_base = """
# SkyWing Airlines - Complete Reference

## Booking Policy
SkyWing offers Light, Flex, and Premium fares. Light fares are non-refundable and do not include checked baggage. Flex fares include one checked bag, free seat selection, and free rebooking up to 3 hours before departure. Premium fares include two checked bags, priority boarding, lounge access, and full flexibility including full refunds.

## Baggage Rules
All passengers: one cabin bag (55x40x20cm, max 8kg) plus one personal item. Checked bags: Light = must purchase separately (from 15 EUR), Flex = one free bag up to 23kg, Premium = two free bags up to 23kg each. Excess weight: 15 EUR/kg for first 9kg over limit, 25 EUR/kg after that. Sports equipment: 30-75 EUR handling fee.

## Loyalty Program - SkyWing Miles
Earn rates: 1 mile/km (Light), 1.5 miles/km (Flex), 2 miles/km (Premium). Tier bonuses: Silver 25%, Gold 50%, Platinum 100%. Qualification: Silver = 25K miles or 30 segments, Gold = 50K miles or 60 segments, Platinum = 100K miles or 120 segments. Benefits by tier: Silver (priority check-in), Gold (+ lounge access, extra bag), Platinum (+ upgrade priority, companion pass).

## Flight Changes and Cancellations
Cancellation more than 24h before departure with ticket bought 7+ days prior: full refund. Less than 24h: fee of 75 EUR (short-haul) or 150 EUR (long-haul). Light fares: no refund, but convertible to 12-month travel credit. Airline-initiated cancellations: free rebooking or full refund.

## Delays and Compensation
Delay over 2h: complimentary refreshments. Over 4h: 15 EUR meal voucher. Overnight: hotel + ground transport for non-local passengers. EU261 compensation: 3h+ delay = 250-600 EUR depending on distance. Missed connections due to SkyWing delays: automatic rebooking on next available flight.

## Special Services
Reduced mobility: request assistance 48h before departure. Wheelchair service, priority boarding, adapted seating at no charge. Service animals: permitted with prior notification. Unaccompanied minors (5-12): supervision service for 50 EUR/segment. Deaf/hard of hearing: text announcements on IFE screen.

## Check-in
Online: opens 24h before, closes 1h (domestic) or 2h (international) before departure. Airport: arrive 2h (domestic) or 3h (international) before departure. Self-service kiosks available at major airports in 12 languages.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_text(knowledge_base)
docs = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(chunks)]

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="agentic_rag",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks.")

## Step 2: Define the Agent State

The agent state holds everything the graph needs as it moves between nodes.

In [ ]:
class AgentState(TypedDict):
    question: str
    context: str
    answer: str
    needs_retrieval: bool
    is_grounded: bool
    retrieval_count: int
    route: str  # "retrieve", "direct", "done"

print("State schema defined.")

## Step 3: Build the Router Node

The router decides whether a question needs retrieval from the knowledge base or can be answered directly.

In [ ]:
router_prompt = ChatPromptTemplate.from_template("""
You are a routing agent for a SkyWing Airlines knowledge base.

Determine if the following question requires looking up information from the airline's knowledge base,
or if it can be answered directly (general knowledge, greetings, math, etc.).

Respond with ONLY one word: "retrieve" or "direct"

Question: {question}

Decision:
""")

def route_question(state: AgentState) -> AgentState:
    """Decide whether to retrieve or answer directly."""
    decision = (router_prompt | llm | StrOutputParser()).invoke(
        {"question": state["question"]}
    ).strip().lower()

    route = "retrieve" if "retrieve" in decision else "direct"
    print(f"  [Router] Question: '{state['question'][:60]}...' -> {route}")

    return {
        **state,
        "route": route,
        "needs_retrieval": route == "retrieve",
    }

print("Router node defined.")

## Step 4: Build the Retrieval Node

In [ ]:
def retrieve(state: AgentState) -> AgentState:
    """Retrieve relevant context from the knowledge base."""
    docs = retriever.invoke(state["question"])
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    count = state.get("retrieval_count", 0) + 1

    print(f"  [Retriever] Retrieved {len(docs)} chunks (attempt {count})")

    return {
        **state,
        "context": context,
        "retrieval_count": count,
    }

print("Retrieval node defined.")

## Step 5: Build the Grounding Evaluator

This is the self-reflection step. After generating an answer, the agent checks whether the answer is actually grounded in the retrieved sources.

In [ ]:
grounding_prompt = ChatPromptTemplate.from_template("""
You are evaluating whether an answer is fully grounded in the provided context.

Context:
{context}

Question: {question}
Answer: {answer}

Evaluate:
1. Is every claim in the answer supported by the context?
2. Does the answer hallucinate any facts not in the context?
3. Is the answer complete given what the context provides?

Respond with ONLY "grounded" or "not_grounded".

Verdict:
""")

def evaluate_grounding(state: AgentState) -> AgentState:
    """Check if the answer is grounded in the retrieved context."""
    verdict = (grounding_prompt | llm | StrOutputParser()).invoke({
        "context": state["context"],
        "question": state["question"],
        "answer": state["answer"],
    }).strip().lower()

    is_grounded = "grounded" in verdict and "not_grounded" not in verdict
    print(f"  [Evaluator] Grounded: {is_grounded}")

    return {
        **state,
        "is_grounded": is_grounded,
    }

print("Grounding evaluator defined.")

## Step 6: Build the Generation Nodes

In [ ]:
rag_answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:
""")

direct_answer_prompt = ChatPromptTemplate.from_template("""
Answer the following question directly. This is a general question that doesn't require
looking up specific airline policy information.

If it's a greeting, respond naturally.
If it's general knowledge, answer concisely.

Question: {question}

Answer:
""")

def generate_rag_answer(state: AgentState) -> AgentState:
    """Generate answer using retrieved context."""
    answer = (rag_answer_prompt | llm | StrOutputParser()).invoke({
        "context": state["context"],
        "question": state["question"],
    })
    print(f"  [Generator] Produced RAG answer ({len(answer)} chars)")
    return {**state, "answer": answer}

def generate_direct_answer(state: AgentState) -> AgentState:
    """Generate answer without retrieval."""
    answer = (direct_answer_prompt | llm | StrOutputParser()).invoke({
        "question": state["question"],
    })
    print(f"  [Generator] Produced direct answer ({len(answer)} chars)")
    return {**state, "answer": answer, "context": "N/A - direct answer"}

print("Generation nodes defined.")

## Step 7: Build the LangGraph

Wire everything together into a state machine.

In [ ]:
def route_after_router(state: AgentState) -> Literal["retrieve", "generate_direct"]:
    """Edge: after routing, go to retrieval or direct answer."""
    return "retrieve" if state["needs_retrieval"] else "generate_direct"

def route_after_evaluation(state: AgentState) -> Literal["done", "retrieve"]:
    """Edge: after evaluation, either finish or re-retrieve."""
    if state["is_grounded"] or state["retrieval_count"] >= 2:
        return "done"
    return "retrieve"

# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("router", route_question)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate_rag", generate_rag_answer)
workflow.add_node("evaluate", evaluate_grounding)
workflow.add_node("generate_direct", generate_direct_answer)

# Set entry point
workflow.set_entry_point("router")

# Add edges
workflow.add_conditional_edges("router", route_after_router)
workflow.add_edge("retrieve", "generate_rag")
workflow.add_edge("generate_rag", "evaluate")
workflow.add_conditional_edges("evaluate", route_after_evaluation, {"done": END, "retrieve": "retrieve"})
workflow.add_edge("generate_direct", END)

# Compile
agent = workflow.compile()

print("Agent graph compiled!")

## Step 8: Test the Agent

Let's test with a mix of questions that should take different paths.

In [ ]:
def run_agent(question):
    """Run the agentic RAG pipeline."""
    initial_state = AgentState(
        question=question,
        context="",
        answer="",
        needs_retrieval=False,
        is_grounded=False,
        retrieval_count=0,
        route="",
    )
    result = agent.invoke(initial_state)
    return result

# Test with different types of questions
test_questions = [
    "Hello, how are you?",                            # Should be direct
    "What is 15 * 23?",                               # Should be direct
    "What is the baggage allowance for Flex tickets?", # Should retrieve
    "How do I earn SkyWing Miles?",                    # Should retrieve
    "What is the capital of France?",                  # Should be direct
    "What compensation do I get for a 5 hour delay?",  # Should retrieve
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    result = run_agent(q)
    print(f"\nFinal answer: {result['answer'][:200]}")
    print(f"Retrievals: {result['retrieval_count']}, Grounded: {result.get('is_grounded', 'N/A')}")

## Step 9: Compare with Always-Retrieve RAG

Let's measure the efficiency gain: how many unnecessary retrievals does the agent avoid?

In [ ]:
import time

def always_retrieve_rag(question):
    """Baseline: always retrieve, always generate."""
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    answer = (rag_answer_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })
    return answer

comparison_questions = [
    "Hello!",                                           # General
    "What is 2 + 2?",                                   # General
    "What is the baggage allowance?",                    # Domain
    "Tell me about loyalty tiers.",                      # Domain
    "What is the weather like in Paris?",                # General
    "How do cancellations work?",                        # Domain
]

print(f"{'Question':<45} {'Agent Route':<15} {'Agent Time':<12} {'Always Time':<12}")
print("-" * 85)

total_agent_retrievals = 0
total_always_retrievals = 0

for q in comparison_questions:
    # Agent
    start = time.time()
    agent_result = run_agent(q)
    agent_time = time.time() - start

    # Always retrieve
    start = time.time()
    always_result = always_retrieve_rag(q)
    always_time = time.time() - start

    route = "retrieve" if agent_result["retrieval_count"] > 0 else "direct"
    total_agent_retrievals += agent_result["retrieval_count"]
    total_always_retrievals += 1

    print(f"{q[:43]:<45} {route:<15} {agent_time:<12.2f} {always_time:<12.2f}")

print(f"\nTotal retrievals - Agent: {total_agent_retrievals}, Always: {total_always_retrievals}")
print(f"Agent saved {total_always_retrievals - total_agent_retrievals} unnecessary retrieval(s).")

## Step 10: Visualize the Agent's Decision Flow

Let's trace the path each question takes through the graph.

In [ ]:
trace_questions = [
    "What is the cancellation fee for long-haul flights?",  # Retrieve + grounded
    "Hi there, what can you help me with?",                  # Direct
    "How many miles do I need for a free flight?",           # Retrieve
]

for q in trace_questions:
    print(f"\n{'='*60}")
    print(f"TRACE: \"{q}\"")
    print(f"{'='*60}")

    result = run_agent(q)

    # Reconstruct the path
    path = ["router"]
    if result["retrieval_count"] > 0:
        for i in range(result["retrieval_count"]):
            path.extend(["retrieve", "generate_rag", "evaluate"])
        path.append("END")
    else:
        path.extend(["generate_direct", "END"])

    print(f"\nPath: {' -> '.join(path)}")
    print(f"Retrievals: {result['retrieval_count']}")
    print(f"Answer: {result['answer'][:150]}")

---

## YOUR TURN: Extend the Agent

Choose one (or more) of these challenges:

### Challenge A: Add Query Rewriting
When the evaluator says the answer is not grounded, instead of re-retrieving with the same query, have the agent rewrite the query to be more specific.

### Challenge B: Add Source Citation
Modify the generator to include inline citations `[1]`, `[2]` etc., mapping to the retrieved chunks.

### Challenge C: Add Confidence Scoring
Have the evaluator return a confidence score (0-1) instead of binary grounded/not_grounded. Only return answers above 0.7 confidence.

In [ ]:
# YOUR TURN: Choose a challenge and implement it below.
# Here's a starting point for Challenge A (query rewriting):

rewrite_prompt = ChatPromptTemplate.from_template("""
The following question was used to search a knowledge base, but the results weren't good enough.
Rewrite the question to be more specific and likely to match relevant documents.

Original question: {question}
Retrieved context (not good enough): {context}

Rewritten question:
""")

def rewrite_query(state: AgentState) -> AgentState:
    """Rewrite the query for better retrieval."""
    new_question = (rewrite_prompt | llm | StrOutputParser()).invoke({
        "question": state["question"],
        "context": state["context"],
    })
    print(f"  [Rewriter] '{state['question'][:40]}...' -> '{new_question[:40]}...'")
    return {**state, "question": new_question}

# TODO: Rebuild the graph with the rewrite node inserted between evaluate and retrieve
# Hint: evaluate -> not_grounded -> rewrite -> retrieve -> generate_rag -> evaluate

print("Implement your chosen challenge above!")

## Key Takeaways

1. **Agentic RAG adds intelligence** to the retrieval decision — not every question needs the knowledge base.
2. **Self-reflection** catches hallucinations by verifying the answer against retrieved sources.
3. **LangGraph** makes it easy to build stateful agent loops with conditional edges.
4. **The trade-off:** more LLM calls (routing, evaluation) vs. better accuracy and efficiency.
5. In production, the router saves cost and latency by skipping retrieval for simple questions.
6. **Query rewriting** on failed retrievals is more effective than blind re-retrieval.

**Next:** In Lab 7 (Capstone), you'll build a production-ready RAG system with hybrid retrieval, re-ranking, and evaluation.